In [ ]:
# Download libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.python.client import device_lib
import gc
import warnings
import os

# Input data files
warnings.filterwarnings("ignore")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Feature engineering
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Family features
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)

    # Title extraction and mapping
    out["Title"] = out["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    title_map = {
        "Mlle":"Miss","Ms":"Miss","Mme":"Mrs","Lady":"Rare","Countess":"Rare",
        "Capt":"Rare","Col":"Rare","Don":"Rare","Dr":"Rare","Major":"Rare",
        "Rev":"Rare","Sir":"Rare","Jonkheer":"Rare","Dona":"Rare"
    }
    out["Title_Type"] = out["Title"].replace(title_map)

    # Cabin & Ticket features
    out["CabinDeck"] = out["Cabin"].str[0].fillna("U")
    out["HasCabin"] = (~out["Cabin"].isna()).astype(int)
    out["TicketPrefix"] = (
        out["Ticket"]
        .str.replace(r"\d+","",regex=True)
        .str.replace(".","",regex=False)
        .str.strip()
        .replace("", "NUM")
    )
    # Group size by ticket
    out["TicketGroupSize"] = out.groupby("Ticket")["Ticket"].transform("count")

    # Fare per person
    out["FarePerPerson"] = (out["Fare"] / out["FamilySize"]).replace([np.inf, -np.inf], np.nan)
    out["FarePerPerson"] = out["FarePerPerson"].clip(upper=100)  # light cap to reduce outlier effect

    # Embarked with imputation
    embarked_mode = out["Embarked"].mode(dropna=True)
    embarked_fill = embarked_mode.iloc[0] if not embarked_mode.empty else "S"
    out["Embarked"] = out["Embarked"].fillna(embarked_fill)

    # Fare grouping
    fare_grp = out.groupby(["Pclass","Embarked"])["Fare"].transform("median")
    out["Fare"] = out["Fare"].fillna(fare_grp)

    # Age grouping
    age_medians = out.groupby(["Sex","Pclass","Title_Type"])["Age"].transform("median")
    out["Age"] = out["Age"].fillna(age_medians)

    # Child / Mother
    out["IsChild"] = (out["Age"] <= 12).astype(int)
    out["IsMother"] = (
        (out["Sex"] == "female") &
        (out["Parch"] > 0) &
        (out["Age"] >= 18) &
        (out["Title_Type"].eq("Mrs"))
    ).astype(int)

    # Interactions
    out["SexBin"] = (out["Sex"] == "male").astype(int)
    out["Age_x_Class"] = out["Age"] * out["Pclass"]
    out["Sex_x_Class"] = out["SexBin"] * out["Pclass"]
    out["HasSib"] = (out["SibSp"] > 0).astype(int)
    out["HasPar"] = (out["Parch"] > 0).astype(int)

    # Name length
    out["NameLen"] = out["Name"].str.len()

    return out

In [ ]:
# Train / test
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test  = pd.read_csv("/kaggle/input/titanic/test.csv")

# Apply feature engineering
train_fe = engineer_features(train)
test_fe  = engineer_features(test)

In [ ]:
# Random seed
np.random.seed(42)
tf.random.set_seed(42)

# Check GPUs
print(device_lib.list_local_devices())
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Fare bins
fare_q, fare_bins = pd.qcut(train_fe["Fare"], q=4, retbins=True, duplicates="drop")
train_fe["FareBin"] = pd.cut(train_fe["Fare"], bins=fare_bins, include_lowest=True, labels=False)
test_fe["FareBin"]  = pd.cut(test_fe["Fare"],  bins=fare_bins, include_lowest=True, labels=False)

# Age bins
age_edges = [0, 12, 20, 40, 60, 120]
train_fe["AgeBin"] = pd.cut(train_fe["Age"], bins=age_edges, include_lowest=True, labels=False)
test_fe["AgeBin"]  = pd.cut(test_fe["Age"],  bins=age_edges, include_lowest=True, labels=False)

# Drop unneeded columns
drop_cols = ["Name","Ticket","Cabin","Title"] 
train_fe = train_fe.drop(columns=drop_cols)
test_fe  = test_fe.drop(columns=drop_cols)

In [ ]:
# Select features
features = [
    # numeric
    "Pclass", "Age", "Fare", "FamilySize", "FarePerPerson", "IsAlone",
    "TicketGroupSize", "HasCabin", "NameLen",
    "IsChild","IsMother",
    "Age_x_Class", "Sex_x_Class",
    # categorical
    "Sex", "Embarked", "Title_Type", "CabinDeck", "TicketPrefix",
    "FareBin", "AgeBin"
]

# Select target
target = "Survived"

# Check for missing features
missing = [c for c in features if c not in train_fe.columns]
if missing:
    print("Missing features (comment out or create before proceeding):", missing)

# Copy into arrays for training
X = train_fe[features].copy()
y = train_fe[target].astype(int).values
X_test = test_fe[features].copy()

# Check bin columns
for col in ["FareBin", "AgeBin"]:
    if col in X.columns:
        X[col] = X[col].astype("category")
        X_test[col] = X_test[col].astype("category")

# Build preprocessing (impute -> encode -> scale)
def make_preprocess():
    # Numeric features
    numeric_features = [
        "Pclass", "Age", "Fare", "FamilySize", "FarePerPerson",
        "IsAlone", "TicketGroupSize", "HasCabin", "NameLen",
        "IsChild","IsMother",
        "Age_x_Class", "Sex_x_Class",
    ]
    # Categorical features
    categorical_features = [
        "Sex", "Embarked", "Title_Type", "CabinDeck", "TicketPrefix",
        "FareBin", "AgeBin"
    ]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # OneHotEncoder
    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    # Build categorical transformer
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", cat_encoder),
    ])

    # Transform
    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, [c for c in numeric_features if c in X.columns]),
            ("cat", categorical_transformer, [c for c in categorical_features if c in X.columns]),
        ],
        remainder="drop"
    )
    return pre

In [ ]:
# Build model
def build_model(input_dim, dropout=0.30, l2_reg=1e-4, lr=1e-3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout * 0.5),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), "accuracy"]
    )
    return model

# Grid search hyperparameters
dropout_grid = [0.20, 0.30, 0.50]
l2_grid      = [1e-5, 1e-4, 1e-3]
lr_grid      = [5e-4, 1e-3, 5e-3]

# Training settings
EPOCHS = 200
BATCH_SIZE = 32
PATIENCE = 20

# K-folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

total = len(dropout_grid) * len(l2_grid) * len(lr_grid)
print(f"Running grid search over {total} combinations...")

# Run grid search
combo_idx = 0
for dr in dropout_grid:
    for l2 in l2_grid:
        for lr in lr_grid:
            combo_idx += 1
            print(f"[{combo_idx}/{total}] dropout={dr}, l2={l2}, lr={lr}")
            aucs, accs = [], []

            for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
                # Fold-specific preprocessing to avoid leakage
                prep = make_preprocess()
                # ✅ Pass y[tr_idx] to fit_transform
                X_tr = prep.fit_transform(X.iloc[tr_idx], y[tr_idx])
                X_va = prep.transform(X.iloc[va_idx])
                y_tr = y[tr_idx]
                y_va = y[va_idx]

                # ✅ Clear TF graph per fold to avoid graph/memory growth
                keras.backend.clear_session()

                model = build_model(X_tr.shape[1], dropout=dr, l2_reg=l2, lr=lr)

                es = callbacks.EarlyStopping(
                    monitor="val_auc", mode="max",
                    patience=PATIENCE, restore_best_weights=True
                )
                
                # Fit model to training data
                model.fit(
                    X_tr, y_tr,
                    validation_data=(X_va, y_va),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=[es],
                    verbose=0
                )

                # Predict on cv data
                va_proba = model.predict(X_va, verbose=0).ravel()
                aucs.append(roc_auc_score(y_va, va_proba))
                accs.append(accuracy_score(y_va, (va_proba >= 0.5).astype(int)))

                # (Optional) cleanup inside the grid loop
                del X_tr, X_va, y_tr, y_va, model
                gc.collect()

            # Save results
            results.append({
                "dropout": dr,
                "l2": l2,
                "lr": lr,
                "mean_auc": float(np.mean(aucs)),
                "std_auc": float(np.std(aucs)),
                "mean_acc": float(np.mean(accs)),
                "std_acc": float(np.std(accs)),
            })

# Rank results and pick the best by mean AUC
res_df = pd.DataFrame(results).sort_values(
    by=["mean_auc", "mean_acc"], ascending=False
).reset_index(drop=True)

print("\nTop 5 configs by CV AUC:")
print(res_df[["dropout", "l2", "lr", "mean_auc", "std_auc", "mean_acc", "std_acc"]]
      .head(5).to_string(index=False))

best = res_df.iloc[0]
best_dropout = float(best["dropout"])
best_l2 = float(best["l2"])
best_lr = float(best["lr"])

print(f"\nBest params -> dropout={best_dropout}, l2={best_l2}, lr={best_lr}")
print(f"CV mean AUC={best['mean_auc']:.4f} (±{best['std_auc']:.4f}), "
      f"mean Acc={best['mean_acc']:.4f} (±{best['std_acc']:.4f})")

# Final training with optimal hyperparameters
oof_proba = np.zeros(len(X), dtype=float)
test_proba_blend = np.zeros(len(X_test), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
    print(f"Final training - fold {fold}/5")

    # Fold-specific preprocessing
    prep = make_preprocess()
    X_tr = prep.fit_transform(X.iloc[tr_idx], y[tr_idx])
    X_va = prep.transform(X.iloc[va_idx])
    X_te = prep.transform(X_test)
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    keras.backend.clear_session()
    model = build_model(
        input_dim=X_tr.shape[1],
        dropout=best_dropout,
        l2_reg=best_l2,
        lr=best_lr
    )

    # Early stopping criteria
    es = callbacks.EarlyStopping(
        monitor="val_auc", mode="max",
        patience=PATIENCE, restore_best_weights=True
    )
    
    # Implement learning rate scheduler for final training
    rlrop = callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, verbose=0
    )

    # Train on full training set
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlrop],
        verbose=0
    )

    # Store OOF and test predictions
    oof_proba[va_idx] = model.predict(X_va, verbose=0).ravel()
    test_proba_blend += model.predict(X_te, verbose=0).ravel() / cv.n_splits

    # Clean up
    del X_tr, X_va, X_te, y_tr, y_va, model, history
    gc.collect()

# Tune classification threshold on OOF for accuracy
grid = np.linspace(0.3, 0.7, 81)
accs = [accuracy_score(y, (oof_proba >= t).astype(int)) for t in grid]
best_thr = float(grid[int(np.argmax(accs))])
oof_acc = max(accs)
oof_auc = roc_auc_score(y, oof_proba)
print(f"OOF AUC={oof_auc:.4f}, OOF ACC={oof_acc:.4f} @ thr={best_thr:.3f}")

In [ ]:
# Bagged predictions -> submission
test_pred = (test_proba_blend >= best_thr).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"].values,
    "Survived": test_pred
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")